# As similar as possible: what does the extra runtime buy?

Suppose you run an aggregation many times: once for every scenario, or inside a search for the
right number of typical days. You want typical days that are as similar as possible to the days
they stand for, and every run costs time. `hierarchical`, `kmeans` and `kmedoids` all group days
by how similar they are.

In this tutorial we compare how close their typical days come and how long they take, first on
one month of hourly data and then on three. By the end you will have seen that

* the three methods come within a few percent of each other,
* their runtimes differ by orders of magnitude, and
* the difference grows much faster than the data: three times the days make `kmedoids` tens of
  times slower.

This is part 1 of two [clustering tutorials](../index.md#clustering). Each part stands on its own.

## 1  One month of hourly data

We take the first 30 days of the example data: solar irradiance, temperature, wind and load,
hour by hour. Every run reduces them to 8 typical days with the same settings, so we write them
down once in a small function; only the clustering `method` will change. `representation="mean"`
is the same for all three methods, so that only the grouping differs, and
`preserve_column_means=False` switches off tsam's rescaling step.

Two numbers tell us what we get for the time. `result.accuracy.weighted_rmse` measures how far
the typical days are from the days they stand for, in normalized units; lower is better.
`result.clustering_duration` measures the clustering step alone, in seconds.

In [ ]:
import numpy as np
import pandas as pd

import tsam
from tsam import ClusterConfig

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
month = raw.iloc[: 30 * 24]


def aggregate(data, method):
    np.random.seed(0)  # kmeans starts from random centers
    return tsam.aggregate(
        data,
        n_clusters=8,
        period_duration="1D",
        cluster=ClusterConfig(method=method, representation="mean"),
        preserve_column_means=False,
    )


def measure(data, method):
    result = aggregate(data, method)
    return {
        "RMSE": result.accuracy.weighted_rmse,
        "seconds": result.clustering_duration,
    }

The first run of each method pays a one-off setup cost that has nothing to do with the method
itself. Run each one once before measuring:

In [ ]:
for method in ["hierarchical", "kmeans", "kmedoids"]:
    aggregate(month, method)

## 2  The default: `hierarchical`

`hierarchical` merges the two most similar groups of days, step by step, until 8 groups are
left:

In [ ]:
scores = {"hierarchical": measure(month, "hierarchical")}
pd.DataFrame(scores).T.round(4)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert scores["hierarchical"]["seconds"] < 0.1

It takes a few milliseconds at most.

## 3  `kmeans`

`kmeans` moves 8 centers until every day is as close as possible to its center, and repeats
this from several random starts:

In [ ]:
scores["kmeans"] = measure(month, "kmeans")
pd.DataFrame(scores).T.round(4)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert scores["kmeans"]["RMSE"] < scores["hierarchical"]["RMSE"]
assert scores["kmeans"]["seconds"] > 10 * scores["hierarchical"]["seconds"]

Its typical days come slightly closer to their days. It needs a fraction of a second instead of
milliseconds.

## 4  `kmedoids`

`kmedoids` chooses 8 real days as centers, and proves that no other choice of 8 days fits better.
It solves an optimization problem to do so:

In [ ]:
scores["kmedoids"] = measure(month, "kmedoids")
pd.DataFrame(scores).T.round(4)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
rmse = [s["RMSE"] for s in scores.values()]
assert max(rmse) / min(rmse) < 1.05
assert scores["kmedoids"]["seconds"] < 3 * scores["kmeans"]["seconds"]

On one month, `kmedoids` is about as fast as `kmeans`, and its typical days fit about as well as
the other two.

## 5  Three times the data

Now take three months, three times as many days, and measure all three methods again:

In [ ]:
quarter = raw.iloc[: 90 * 24]
longer = {method: measure(quarter, method) for method in scores}

pd.concat(
    {"30 days": pd.DataFrame(scores).T, "90 days": pd.DataFrame(longer).T}, axis=1
).round(4)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert longer["kmedoids"]["seconds"] > 10 * scores["kmedoids"]["seconds"]
assert longer["kmeans"]["seconds"] < 3 * scores["kmeans"]["seconds"]
assert longer["hierarchical"]["seconds"] < 0.1
rmse = [s["RMSE"] for s in longer.values()]
assert max(rmse) / min(rmse) < 1.05

The fit hardly changes, and neither does the runtime of `hierarchical` and `kmeans`. `kmedoids`,
however, has become tens of times slower: three times the days, but far more than three times
the work. The problem it solves grows with the square of the number of days, so its runtime
grows much faster than the data.

## 6  Try it yourself

The difference between the methods grows nonlinearly with the number of periods, so a test on a
small sample tells you little about a large one. See it for yourself: change `N_DAYS` and run the
cell again. On our machine, `kmedoids` needs about half a minute for 180 days, and minutes for a
whole year.

In [ ]:
N_DAYS = 60  # try 90, 120 or 180

data = raw.iloc[: N_DAYS * 24]
pd.DataFrame({method: measure(data, method) for method in scores}).T.round(4)

What counts is the number of periods, not the number of hours: with weekly periods, a whole year
has only 52. [How long will this take?](../../how-to/runtime.ipynb) shows how to use that.

## What you've learned

* `hierarchical`, `kmeans` and `kmedoids` all group days by similarity, and their typical days
  come within a few percent of each other.
* Their runtimes differ by orders of magnitude: milliseconds, a fraction of a second, and seconds
  or more.
* The difference grows nonlinearly with the number of periods. Three times the days made
  `kmedoids` tens of times slower, while the other two barely changed.
* What the extra time of `kmedoids` buys is an optimal choice of real days as centers, not a
  closer fit.

Where to go next:

* [How long will this take?](../../how-to/runtime.ipynb) — measure runtime on your data, and
  shorten it.
* [Clustering methods](../../how-to/clustering_methods.ipynb) — pick a method by what your model
  needs.
* [Part 2: Seasonal storage](02_seasonal_storage.ipynb) — when grouping by similarity is the
  wrong goal.